# PaliGemma 2 ile küçük deney: dil ve çözünürlük
Bu notebook iki soruyu ölçer:
1. **Dil:** Aynı soruyu Türkçe ve İngilizce sorunca doğruluk değişiyor mu? (`mix-224` modeli)
2. **Çözünürlük:** Tabela/fiş okutmada 224px ile 448px arasında fark var mı? (`mix-224` ve `mix-448`)

**Başlamadan önce**
- Runtime > Change runtime type > **G4 GPU**
- Hugging Face'te [google/paligemma2-3b-mix-224](https://huggingface.co/google/paligemma2-3b-mix-224) ve [mix-448](https://huggingface.co/google/paligemma2-3b-mix-448) sayfalarında lisansı kabul et.
- Hugging Face'ten bir **read token** oluştur, Colab'da sol menüdeki 🔑 *Secrets* bölümüne `HF_TOKEN` adıyla ekle.
- 15-20 fotoğrafı `veri/` klasörüne yükle ve `etiketler.csv` dosyasını hazırla (aşağıda format var).

In [6]:
!pip -q install -U transformers accelerate pandas

In [7]:
import torch, time, re, unicodedata, difflib
import pandas as pd
from PIL import Image
from google.colab import userdata
from huggingface_hub import login
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor

login(userdata.get("HF_TOKEN"))
print(torch.__version__, torch.cuda.get_device_name(0))

2.11.0+cu130 NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1. Veri: `etiketler.csv`
Her satır bir örnek. Sütunlar:

| sütun | anlamı |
|---|---|
| `dosya` | `veri/` içindeki görsel adı |
| `gorev` | `vqa` (soru-cevap) veya `ocr` (yazı okuma) |
| `soru_tr` | Türkçe soru (vqa için) |
| `soru_en` | Aynı sorunun İngilizcesi (vqa için) |
| `cevap` | **Senin yazdığın** doğru cevap / görseldeki yazı |

İpucu: vqa için kısa cevaplı sorular seç ("Tabelada hangi sayı yazıyor?", "Fişin toplamı kaç?"). Etiketleri kendin yaz, modele yazdırma.

In [8]:
%%writefile etiketler.csv
dosya,gorev,soru_tr,soru_en,cevap
tabela1.jpg,ocr,,,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalaman Antalya
tabela2.jpg,ocr,,,Eskişehir Ayrancı Dikmen Kızılay
tabela3.jpg,ocr,,,Tayakadın Karaburun Çatalca İstanbul Havalimanı Göktürk Kemerburgaz Hasdal Arnavutköy
tabela4.jpg,ocr,,,Polatlı Ankara Sivrihisar Eskişehir Mihalıççık
tabela5.jpg,ocr,,,Büyük Çamlıca Cami Çevreyolu Üsküdar Kısıklı Rasathane Ümraniye
tabela6.jpg,ocr,,,Agora Kemeraltı Liman (Harbour) Hastaneler Nazım Hikmet Kültür Merkezi Konak Metro İstasyonu Sanat Galerisi
tabela7.jpg,ocr,,,Otogar Havalimanı Ordu Giresun Trabzon
tabela8.jpg,ocr,,,Yeşilköy Florya Sahilyolu Havalimanı Çevreyolu
tabela9.jpg,ocr,,,Kağıthane İstanbul Yeni Havalimanı Fatih Sultan Mehmet Köp. 2. Çevreyolu Yavuz Sultan Selim Köp.
tabela10.jpg,ocr,,,Kızılay Söğütözü Eskişehir Konya
fis1.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"82,55"
fis2.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"2.685,00"
fis3.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"304.31"
fis4.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"26,15"
fis5.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"216,49"
fis6.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"178,90"
fis7.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"34,75"
fis8.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"248,85"
fis9.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"450,00"
fis10.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"128,95"
nota1.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,8
nota2.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,10
nota3.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,7
nota4.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,9
nota5.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,7
nota6.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,6
nota7.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,12
hayvan1.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
hayvan2.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
hayvan3.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
hayvan4.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
hayvan5.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,5
hayvan6.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
hayvan7.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,3
hayvan8.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,3
hayvan9.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,1
hayvan10.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,4
rontgen1.jpg,vqa,"Görseldeki el röntgeninde kaç tane vida var?","How many screws are there in the hand X-ray shown in the image?",5
rontgen2.jpg,vqa,"Ağız röntgeninde kaç adet implant var?","How many implants are there in the dental X-ray?",8
rontgen3.jpg,vqa,"Görseldeki kol röntgeninde kaç tane vida var?","How many screws are there in the arm X-ray shown in the image?",6
rontgen4.jpg,vqa,"Görseldeki kol röntgeninde kaç tane vida var?","How many screws are there in the arm X-ray shown in the image?",12
rontgen5.jpg,vqa,"Ağız röntgeninde kaç adet diş dolgusu var?","How many dental fillings are there on the dental X-ray?",8

Overwriting etiketler.csv


In [9]:
# İlk çalıştırmada örnek şablon oluşturur. Kendi verinle değiştir.
import os
os.makedirs("veri", exist_ok=True)
if not os.path.exists("etiketler.csv"):
    pd.DataFrame([
        {"dosya":"tabela1.jpg","gorev":"ocr","soru_tr":"","soru_en":"","cevap":"ECZANE"},
        {"dosya":"fis1.jpg","gorev":"vqa","soru_tr":"Fişin toplam tutarı nedir?","soru_en":"What is the total amount on the receipt?","cevap":"84,50"},
    ]).to_csv("etiketler.csv", index=False)
df = pd.read_csv("etiketler.csv").fillna("")
print(len(df), "örnek"); df.head()

42 örnek


,dosya,gorev,soru_tr,soru_en,cevap
0,tabela1.jpg,ocr,,,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalama...
1,tabela2.jpg,ocr,,,Eskişehir Ayrancı Dikmen Kızılay
2,tabela3.jpg,ocr,,,Tayakadın Karaburun Çatalca İstanbul Havaliman...
3,tabela4.jpg,ocr,,,Polatlı Ankara Sivrihisar Eskişehir Mihalıççık
4,tabela5.jpg,ocr,,,Büyük Çamlıca Cami Çevreyolu Üsküdar Kısıklı R...


## 2. Yardımcı fonksiyonlar

In [10]:
def normalize(s):
    # Türkçe I/İ sorununu önlemek için önce özel harfleri çevir, sonra küçült
    s = s.replace("İ","i").replace("I","ı")
    s = unicodedata.normalize("NFKC", s).casefold()
    s = re.sub(r"[^\w\s,.]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def levenshtein(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def cer(pred, ref):
    pred, ref = normalize(pred), normalize(ref)
    return levenshtein(pred, ref) / max(len(ref), 1)

SAYI_HARITASI = {
    "0": ["sıfır", "zero"], "1": ["bir", "one"], "2": ["iki", "two"],
    "3": ["üç", "three"], "4": ["dört", "four"], "5": ["beş", "five"],
    "6": ["altı", "six"], "7": ["yedi", "seven"], "8": ["sekiz", "eight"],
    "9": ["dokuz", "nine"], "10": ["on", "ten"], "12": ["on iki", "twelve"]
}

def contains_match(pred, ref):
    p = normalize(pred).replace(",", ".")
    r = normalize(ref).replace(",", ".")

    # 1. Doğrudan kelime/sayı sınırıyla eşleşme (\b2\b gibi)
    pattern = rf"\b{re.escape(r)}\b"
    if re.search(pattern, p):
        return True

    # 2. Rakamın yazıyla karşılığı çıktı içinde geçiyor mu?
    if r in SAYI_HARITASI:
        for kelime in SAYI_HARITASI[r]:
            if re.search(rf"\b{re.escape(kelime)}\b", p):
                return True

    return False

from collections import Counter

def parse_words(text):
    """
    Metni Türkçe karakter uyumlu şekilde küçültür, noktalama işaretlerini
    temizler ve kelime listesi (token list) olarak döner.
    """
    text = text.replace("İ", "i").replace("I", "ı")
    text = unicodedata.normalize("NFKC", text).casefold()
    # Harf ve rakam haricindeki karakterleri boşluğa çevir
    text = re.sub(r"[^\w\s]|_", " ", text)
    return text.strip().split()

def kelime_basarimi(pred, ref):
    """
    Referanstaki kelimelerin model çıktısında bulunma oranını ölçer (sıradan bağımsız).
    Tekrar eden kelimeler Counter ile güvenli şekilde kontrol edilir.
    Dönüş: (başarı_yüzdesi, eşleşen_kelime_sayısı, referans_kelime_sayısı)
    """
    p_words = parse_words(pred)
    r_words = parse_words(ref)

    toplam = len(r_words)
    if toplam == 0:
        return 100.0, 0, 0

    p_counter = Counter(p_words)
    r_counter = Counter(r_words)

    # Referans ile tahminin ortak kelime kesişimi (min sayımı alır, tekrarı suistimal etmez)
    eslesen = sum((p_counter & r_counter).values())
    skor_yuzde = round((eslesen / toplam) * 100, 2)

    return skor_yuzde, eslesen, toplam

In [11]:
def load(model_id):
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="cuda").eval()
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor

@torch.inference_mode()
def ask(model, processor, image, prompt, max_new_tokens=60):
    full_prompt = f"<image>{prompt}"
    inputs = processor(text=full_prompt, images=image, return_tensors="pt").to(torch.bfloat16).to(model.device)
    n_in = inputs["input_ids"].shape[-1]          # resim token'ları + metin
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    text = processor.decode(out[0][n_in:], skip_special_tokens=True)
    return text.strip(), n_in, dt

## Deney 1: Türkçe soru mu, İngilizce soru mu? (224px)
PaliGemma "mix" modelleri istem başında görev ve dil kodu bekler: `answer en ...` / `answer tr ...`.
Soru metni Türkçe, kod da `tr` olursa fark ediyor mu? İstersen üçüncü bir koşul da dene: Türkçe soru + `answer en` kodu.

In [12]:
model, proc = load("google/paligemma2-3b-mix-224")
vqa = df[df.gorev=="vqa"]
rows = []
for _, r in vqa.iterrows():
    img = Image.open(f"veri/{r.dosya}").convert("RGB")
    for kosul, prompt in [("en", f"answer en {r.soru_en}"),
                          ("tr", f"answer tr {r.soru_tr}")]:
        pred, n_in, dt = ask(model, proc, img, prompt)
        rows.append(dict(dosya=r.dosya, kosul=kosul, tahmin=pred, cevap=r.cevap,
                         dogru=contains_match(pred, r.cevap), token=n_in, sn=round(dt,2)))
res1 = pd.DataFrame(rows)
res1

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

,dosya,kosul,tahmin,cevap,dogru,token,sn
0,fis1.jpg,en,82.55,"82,55",True,269,2.77
1,fis1.jpg,tr,"82,55","82,55",True,269,0.07
2,fis2.jpg,en,19.00,"2.685,00",False,269,0.07
3,fis2.jpg,tr,7.95,"2.685,00",False,269,0.06
4,fis3.jpg,en,304.31,304.31,True,269,0.09
...,...,...,...,...,...,...,...
59,rontgen3.jpg,tr,Üç.,6,False,272,0.05
60,rontgen4.jpg,en,Two.,12,False,276,0.04
61,rontgen4.jpg,tr,Üç.,12,False,272,0.05
62,rontgen5.jpg,en,32,8,False,273,0.04


In [13]:
print(res1.groupby("kosul").dogru.agg(["mean","sum","count"]).rename(
    columns={"mean":"doğruluk","sum":"doğru","count":"toplam"}))

       doğruluk  doğru  toplam
kosul                         
en      0.50000     16      32
tr      0.46875     15      32


## Deney 2: 224px mi, 448px mi? (OCR)
Aynı görseller, aynı istem (`ocr`), iki farklı çözünürlük. Metrik: karakter hata oranı (CER, düşük iyi). İstem başındaki token sayısını da kaydediyoruz (224px için 256 resim token'ı, 448px için 1024 olmalı).

In [14]:
import gc

ocr = df[df.gorev == "ocr"]
rows = []

for model_id, res_adi in [("google/paligemma2-3b-mix-224", "224"),
                          ("google/paligemma2-3b-mix-448", "448")]:
    # GPU belleğini temizle
    if "model" in locals():
        del model, proc
        gc.collect()
        torch.cuda.empty_cache()

    model, proc = load(model_id)

    for _, r in ocr.iterrows():
        img = Image.open(f"veri/{r.dosya}").convert("RGB")
        # Uzun tabelalarda metnin kesilmemesi için token sınırını 128 yaptık
        pred, n_in, dt = ask(model, proc, img, "ocr", max_new_tokens=128)

        # Kelime bazlı sıradan bağımsız değerlendirme
        basari_yuzde, eslesen, toplam = kelime_basarimi(pred, r.cevap)

        rows.append(dict(
            dosya=r.dosya,
            cozunurluk=res_adi,
            tahmin=pred,
            cevap=r.cevap,
            tam_dogru=normalize(pred) == normalize(r.cevap),  # Eksik olan sütun eklendi
            eslesen_kelime=eslesen,
            toplam_kelime=toplam,
            kelime_basarimi_yuzde=basari_yuzde,
            cer=round(cer(pred, r.cevap), 3),
            token=n_in,
            sn=round(dt, 2)
        ))

res2 = pd.DataFrame(rows)
res2.head(10)

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/75.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

,dosya,cozunurluk,tahmin,cevap,tam_dogru,eslesen_kelime,toplam_kelime,kelime_basarimi_yuzde,cer,token,sn
0,tabela1.jpg,224,Marmaris\nUlu Muğla İzmir\nKayseri\nKayseri\nA...,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalama...,False,4,8,50.00,0.309,259,0.24
1,tabela2.jpg,224,Eskişehir\nAyranci\nDikmen\nKızılay,Eskişehir Ayrancı Dikmen Kızılay,False,3,4,75.00,0.031,259,0.18
2,tabela3.jpg,224,Taysladın Karabük\nŞofala\nİstanbul Havalimanı...,Tayakadın Karaburun Çatalca İstanbul Havaliman...,False,4,9,44.44,0.224,259,0.37
3,tabela4.jpg,224,Polatlı Ankara\nSıvrihisar\nEskişehir\nMihalıççık,Polatlı Ankara Sivrihisar Eskişehir Mihalıççık,False,4,5,80.00,0.022,259,0.24
4,tabela5.jpg,224,Büyük Camlıca Camii\nÇevreyolu Üsküdar\nKısıkl...,Büyük Çamlıca Cami Çevreyolu Üsküdar Kısıklı R...,False,6,8,75.00,0.032,259,0.31
5,tabela6.jpg,224,Agora\nKemeralti\nLiman (Harbour)\nHastaneler\...,Agora Kemeraltı Liman (Harbour) Hastaneler Naz...,False,12,14,85.71,0.019,259,0.41
6,tabela7.jpg,224,Otogar\nHavalımani\nOrdu\nGiresun\nTrabzon,Otogar Havalimanı Ordu Giresun Trabzon,False,4,5,80.00,0.053,259,0.20
7,tabela8.jpg,224,Yeşilköy F.Lorya\nSahil yolu\n→Hava limani\nÇe...,Yeşilköy Florya Sahilyolu Havalimanı Çevreyolu,False,2,5,40.00,0.152,259,0.29
8,tabela9.jpg,224,Kağıthane\nİstanbul Yeni Havalimanı Fatih Sult...,Kağıthane İstanbul Yeni Havalimanı Fatih Sulta...,True,14,14,100.00,0.000,259,0.38
9,tabela10.jpg,224,Kızılay\nSoğutçu\n↑ Eskisehir\nKonya,Kızılay Söğütözü Eskişehir Konya,False,2,4,50.00,0.188,259,0.20


In [15]:
print(res2.groupby("cozunurluk").agg(ort_CER=("cer","mean"), tam_dogru=("tam_dogru","sum"),
      ort_token=("token","mean"), ort_sn=("sn","mean"), n=("cer","count")).round(3))

            ort_CER  tam_dogru  ort_token  ort_sn   n
cozunurluk                                           
224           0.103          1      259.0   0.282  10
448           0.027          5     1027.0   0.301  10


## Sonuçları kaydet
Tabloları yazına koy, `sonuclar_*.csv` dosyalarını GitHub'a ekle. Hatalı tahminlere tek tek bak: sayıyı mı karıştırdı, Türkçe karakteri mi (ğ, ş, ı) bozdu, yoksa yazıyı hiç görmedi mi?

In [16]:
# 1. VQA sonuçlarını kaydet
res1.to_csv("sonuclar_dil.csv", index=False)

# 2. OCR Çözünürlük bazında özet istatistik tablosu
ozet_cozunurluk = res2.groupby("cozunurluk").agg(
    ort_kelime_basarimi=("kelime_basarimi_yuzde", "mean"),
    tam_dogru_sayisi=("tam_dogru", "sum"),
    toplam_eslesen_kelime=("eslesen_kelime", "sum"),
    toplam_referans_kelime=("toplam_kelime", "sum"),
    ort_CER=("cer", "mean"),
    ort_token=("token", "mean"),
    ort_sn=("sn", "mean"),
    ornek_sayisi=("dosya", "count")
).round(2)

print("=== ÇÖZÜNÜRLÜK KARŞILAŞTIRMA ÖZETİ ===")
print(ozet_cozunurluk)

# 3. OCR detay ve özet tablolarını kaydet
res2.to_csv("sonuclar_cozunurluk.csv", index=False)
ozet_cozunurluk.to_csv("sonuclar_cozunurluk_ozet.csv")

print("\nTüm dosyalar başarıyla kaydedildi:")
print("- sonuclar_dil.csv")
print("- sonuclar_cozunurluk.csv")
print("- sonuclar_cozunurluk_ozet.csv")

=== ÇÖZÜNÜRLÜK KARŞILAŞTIRMA ÖZETİ ===
            ort_kelime_basarimi  tam_dogru_sayisi  toplam_eslesen_kelime  \
cozunurluk                                                                 
224                       68.02                 1                     55   
448                       87.17                 5                     69   

            toplam_referans_kelime  ort_CER  ort_token  ort_sn  ornek_sayisi  
cozunurluk                                                                    
224                             76     0.10      259.0    0.28            10  
448                             76     0.03     1027.0    0.30            10  

Tüm dosyalar başarıyla kaydedildi:
- sonuclar_dil.csv
- sonuclar_cozunurluk.csv
- sonuclar_cozunurluk_ozet.csv
